# Mount Drive and Check GPU

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
# This lists everything in your main Drive folder
import os
path = "/content/drive/MyDrive/"
print("Folders found in MyDrive:")
print(os.listdir(path))

Folders found in MyDrive:
['Desktop 2018.04.14 - 22.31.34.06.mp4', 'Desktop 2018.04.22 - 20.54.04.03.mp4', 'The Motivation Myth by Jeff Haden.epub', 'Emailing CamScanner 04-20-2020 15.09.40-1.gdoc', '20200806_052602.jpg', '20200805_215559.jpg', '20200806_052633.jpg', '20200806_052639.jpg', '20200806_052945.jpg', '20200806_052940.jpg', '20200806_053048.jpg', '20200806_103347.jpg', '20200806_103413.jpg', '20200806_103418.jpg', '20200808_100903.jpg', '20200808_100917.jpg', '20200808_100938.jpg', '20200808_100941.jpg', '20200808_101120.jpg', '20200808_200909.jpg', '20200809_111356.jpg', '20200808_200915.jpg', '20200809_111343.jpg', '20200809_184258.jpg', '20200809_184302.jpg', '20200809_184307.jpg', '20200809_184311.jpg', '20200810_102345.jpg', '20200810_102347.jpg', '20200810_102352.jpg', '20200810_102420.jpg', '20200810_102429.jpg', '20200810_102430.jpg', '20200810_102431.jpg', 'BASIC_MATHS_FORMULAE.pdf', 'Nord VPN Client v6.23.11.0.rar', 'Desktop Screenshot 2020.10.11 - 02.04.29.29.png'

In [3]:
# Replace 'Exact Folder Name' with what appeared in the list above
correct_path = '/content/drive/MyDrive/DATASET--ANTS'

if os.path.exists(correct_path):
    files = os.listdir(correct_path)
    print(f"Success! Files found: {files}")
else:
    print("Path still not found. Check for typos or hidden spaces.")

Success! Files found: ['Road Traffic - Dataset 01.mp4', 'Road Traffic - Dataset 02.mp4', 'outputs']


In [4]:
import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))
    print("CUDA device count:", torch.cuda.device_count())
else:
    print("GPU is not connected.")

CUDA available: True
GPU name: Tesla T4
CUDA device count: 1


In [5]:
!nvidia-smi

Fri Apr 24 10:01:58 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   55C    P8             13W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

# Install Dependencies

In [6]:
!pip install -q ultralytics opencv-python pandas numpy tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 25.9 MB/s eta 0:00:00


# Create Project Folder

In [7]:
import os

PROJECT_DIR = "/content/smart_drone_colab"
os.makedirs(PROJECT_DIR, exist_ok=True)

print("Project folder created:", PROJECT_DIR)

Project folder created: /content/smart_drone_colab


# config.py

In [8]:
%%writefile /content/smart_drone_colab/config.py
import os


# =========================
# Dataset paths
# =========================

DATASET_DIR = "/content/drive/MyDrive/DATASET--ANTS"

VIDEO_1 = os.path.join(DATASET_DIR, "Road Traffic - Dataset 01.mp4")
VIDEO_2 = os.path.join(DATASET_DIR, "Road Traffic - Dataset 02.mp4")


# =========================
# Output paths
# =========================

OUTPUT_DIR = "/content/drive/MyDrive/DATASET--ANTS/outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)


# =========================
# Model settings
# =========================

# Corrected filename from yolov26n.pt to yolo26m.pt
MODEL_PATH = "/content/yolo26m.pt"

IMAGE_SIZE = 640
CONF_THRESHOLD = 0.25
IOU_THRESHOLD = 0.45


# =========================
# Vehicle class IDs from COCO
# =========================

VEHICLE_CLASS_IDS = {
    2: "car",
    3: "motorcycle",
    5: "bus",
    6: "train",
    7: "truck"
}
# =========================
# Processing settings
# =========================

FRAME_SKIP = 1
DRAW_TRAILS = True
MAX_TRAIL_LENGTH = 30

# =========================
# Optional class correction settings
# =========================

ENABLE_BUS_TO_TRAIN_HEURISTIC = True
BUS_TO_TRAIN_ASPECT_RATIO_THRESHOLD = 3.0
BUS_TO_TRAIN_MIN_AREA = 25000

Writing /content/smart_drone_colab/config.py


# file_utils.py

In [9]:
%%writefile /content/smart_drone_colab/file_utils.py
import os
import json


def ensure_dir(path: str):
    os.makedirs(path, exist_ok=True)


def save_json(data: dict, output_path: str):
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=4)


def get_video_name_without_ext(video_path: str) -> str:
    filename = os.path.basename(video_path)
    name, _ = os.path.splitext(filename)
    return name.replace(" ", "_").replace("-", "_")

Writing /content/smart_drone_colab/file_utils.py


# report_generator.py

In [10]:
%%writefile /content/smart_drone_colab/report_generator.py
import pandas as pd


def save_detection_report(rows, output_csv_path: str):
    """
    Saves frame-level tracking/detection data.

    Each row contains:
    - track_id
    - vehicle_type
    - frame_number
    - timestamp_seconds
    - confidence
    - bbox coordinates
    - event
    """

    columns = [
        "track_id",
        "vehicle_type",
        "frame_number",
        "timestamp_seconds",
        "confidence",
        "x1",
        "y1",
        "x2",
        "y2",
        "event"
    ]

    df = pd.DataFrame(rows, columns=columns)
    df.to_csv(output_csv_path, index=False)

    return output_csv_path


def build_summary(counted_tracks, processing_duration_seconds: float, total_frames: int, fps: float):
    """
    Builds final summary for the video.
    """

    vehicle_type_breakdown = {}

    for track_id, data in counted_tracks.items():
        vehicle_type = data["vehicle_type"]
        vehicle_type_breakdown[vehicle_type] = vehicle_type_breakdown.get(vehicle_type, 0) + 1

    summary = {
        "total_vehicle_count": len(counted_tracks),
        "vehicle_type_breakdown": vehicle_type_breakdown,
        "processing_duration_seconds": round(processing_duration_seconds, 2),
        "total_frames": total_frames,
        "fps": fps,
        "counting_method": "Each unique ByteTrack tracking ID is counted only once."
    }

    return summary

Writing /content/smart_drone_colab/report_generator.py


# visualizer.py

In [11]:
%%writefile /content/smart_drone_colab/visualizer.py
import cv2
from collections import defaultdict

from config import DRAW_TRAILS, MAX_TRAIL_LENGTH


track_history = defaultdict(list)


def draw_box(frame, bbox, track_id, vehicle_type, confidence):
    x1, y1, x2, y2 = map(int, bbox)

    label = f"ID {track_id} | {vehicle_type} | {confidence:.2f}"

    cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)

    label_y = max(y1 - 10, 20)
    cv2.putText(
        frame,
        label,
        (x1, label_y),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.55,
        (0, 255, 0),
        2,
        cv2.LINE_AA
    )


def draw_trail(frame, bbox, track_id):
    if not DRAW_TRAILS:
        return

    x1, y1, x2, y2 = map(int, bbox)
    center_x = int((x1 + x2) / 2)
    center_y = int((y1 + y2) / 2)

    track_history[track_id].append((center_x, center_y))

    if len(track_history[track_id]) > MAX_TRAIL_LENGTH:
        track_history[track_id].pop(0)

    points = track_history[track_id]

    for i in range(1, len(points)):
        cv2.line(frame, points[i - 1], points[i], (255, 255, 0), 2)


def draw_summary_overlay(frame, total_count, vehicle_type_breakdown, progress_percent=None):
    y = 30

    cv2.putText(
        frame,
        f"Total Unique Vehicles: {total_count}",
        (20, y),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.8,
        (0, 0, 255),
        2,
        cv2.LINE_AA
    )

    y += 35

    breakdown_text = " | ".join([
        f"{vehicle_type}: {count}"
        for vehicle_type, count in vehicle_type_breakdown.items()
    ])

    if breakdown_text:
        cv2.putText(
            frame,
            breakdown_text,
            (20, y),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.65,
            (0, 0, 255),
            2,
            cv2.LINE_AA
        )

    if progress_percent is not None:
        y += 35
        cv2.putText(
            frame,
            f"Progress: {progress_percent:.1f}%",
            (20, y),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.65,
            (255, 0, 0),
            2,
            cv2.LINE_AA
        )

Writing /content/smart_drone_colab/visualizer.py


# video_processor.py

In [12]:
%%writefile /content/smart_drone_colab/video_processor.py
import os
import time
import cv2
import torch
from tqdm import tqdm
from ultralytics import YOLO

from config import (
    MODEL_PATH,
    IMAGE_SIZE,
    CONF_THRESHOLD,
    IOU_THRESHOLD,
    VEHICLE_CLASS_IDS,
    FRAME_SKIP,
    ENABLE_BUS_TO_TRAIN_HEURISTIC,
    BUS_TO_TRAIN_ASPECT_RATIO_THRESHOLD,
    BUS_TO_TRAIN_MIN_AREA
)

from report_generator import save_detection_report, build_summary
from file_utils import ensure_dir, save_json
from visualizer import draw_box, draw_trail, draw_summary_overlay


def get_device():
    if torch.cuda.is_available():
        return 0
    return "cpu"


def print_gpu_status(prefix=""):
    if torch.cuda.is_available():
        allocated_mb = torch.cuda.memory_allocated(0) / 1024**2
        reserved_mb = torch.cuda.memory_reserved(0) / 1024**2

        print(
            f"{prefix}GPU memory | "
            f"allocated: {allocated_mb:.2f} MB | "
            f"reserved: {reserved_mb:.2f} MB"
        )


def maybe_correct_vehicle_class(class_id, vehicle_type, bbox):
    """
    Fixes a common drone-footage issue:
    trains may be visually confused as buses from top view.

    This does NOT change the detector itself.
    It only post-processes the label for reporting/counting.
    """

    if not ENABLE_BUS_TO_TRAIN_HEURISTIC:
        return class_id, vehicle_type

    # Only correct bus -> train
    if vehicle_type != "bus":
        return class_id, vehicle_type

    x1, y1, x2, y2 = bbox
    width = abs(x2 - x1)
    height = abs(y2 - y1)

    if width <= 0 or height <= 0:
        return class_id, vehicle_type

    aspect_ratio = max(width / height, height / width)
    area = width * height

    if aspect_ratio >= BUS_TO_TRAIN_ASPECT_RATIO_THRESHOLD and area >= BUS_TO_TRAIN_MIN_AREA:
        return 6, "train"

    return class_id, vehicle_type


def process_video(
    input_video_path: str,
    output_video_path: str,
    output_csv_path: str,
    output_summary_path: str
):
    if not os.path.exists(input_video_path):
        raise FileNotFoundError(f"Input video not found: {input_video_path}")

    ensure_dir(os.path.dirname(output_video_path))
    ensure_dir(os.path.dirname(output_csv_path))
    ensure_dir(os.path.dirname(output_summary_path))

    print("=" * 80)
    print("Loading model:", MODEL_PATH)
    print("Torch CUDA available:", torch.cuda.is_available())

    if torch.cuda.is_available():
        print("GPU name:", torch.cuda.get_device_name(0))
        print("Current CUDA device:", torch.cuda.current_device())
        print("Using YOLO device: cuda:0")
        device = 0
    else:
        print("Using YOLO device: CPU")
        device = "cpu"

    model = YOLO(MODEL_PATH)

    if torch.cuda.is_available():
        model.to("cuda")

    print("Model loaded.")
    print_gpu_status(prefix="After model load | ")
    print("=" * 80)

    cap = cv2.VideoCapture(input_video_path)

    if not cap.isOpened():
        raise RuntimeError(f"Could not open video: {input_video_path}")

    fps = cap.get(cv2.CAP_PROP_FPS)
    original_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    original_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    print("Video information:")
    print("FPS:", fps)
    print("Width:", original_width)
    print("Height:", original_height)
    print("Total frames:", total_frames)
    print("Frame skip:", FRAME_SKIP)
    print("Image size:", IMAGE_SIZE)
    print("Confidence threshold:", CONF_THRESHOLD)
    print("IOU threshold:", IOU_THRESHOLD)

    fourcc = cv2.VideoWriter_fourcc(*"mp4v")

    writer = cv2.VideoWriter(
        output_video_path,
        fourcc,
        fps,
        (original_width, original_height)
    )

    if not writer.isOpened():
        cap.release()
        raise RuntimeError(f"Could not create output video writer: {output_video_path}")

    counted_tracks = {}
    report_rows = []

    frame_number = 0
    start_time = time.time()

    print("Starting video processing...")

    progress_bar = tqdm(total=total_frames)

    while True:
        ret, frame = cap.read()

        if not ret:
            break

        frame_number += 1
        progress_bar.update(1)

        timestamp_seconds = frame_number / fps if fps > 0 else 0

        if FRAME_SKIP > 1 and frame_number % FRAME_SKIP != 0:
            vehicle_type_breakdown = {}

            for _, data in counted_tracks.items():
                vehicle_type = data["vehicle_type"]
                vehicle_type_breakdown[vehicle_type] = vehicle_type_breakdown.get(vehicle_type, 0) + 1

            progress_percent = (frame_number / total_frames) * 100 if total_frames > 0 else 0

            draw_summary_overlay(
                frame,
                total_count=len(counted_tracks),
                vehicle_type_breakdown=vehicle_type_breakdown,
                progress_percent=progress_percent
            )

            writer.write(frame)
            continue

        results = model.track(
            frame,
            persist=True,
            tracker="bytetrack.yaml",
            imgsz=IMAGE_SIZE,
            conf=CONF_THRESHOLD,
            iou=IOU_THRESHOLD,
            verbose=False,
            device=device
        )

        if torch.cuda.is_available() and frame_number % 500 == 0:
            print_gpu_status(prefix=f"Frame {frame_number} | ")

        if results is None or len(results) == 0:
            writer.write(frame)
            continue

        result = results[0]

        if result.boxes is None:
            writer.write(frame)
            continue

        boxes = result.boxes

        if boxes.id is None:
            writer.write(frame)
            continue

        xyxy_list = boxes.xyxy.cpu().numpy()
        conf_list = boxes.conf.cpu().numpy()
        cls_list = boxes.cls.cpu().numpy()
        id_list = boxes.id.cpu().numpy()

        for bbox, confidence, class_id, track_id in zip(
            xyxy_list,
            conf_list,
            cls_list,
            id_list
        ):
            class_id = int(class_id)
            track_id = int(track_id)
            confidence = float(confidence)

            if class_id not in VEHICLE_CLASS_IDS:
                continue

            vehicle_type = VEHICLE_CLASS_IDS[class_id]
            x1, y1, x2, y2 = bbox.tolist()

            class_id, vehicle_type = maybe_correct_vehicle_class(
                class_id=class_id,
                vehicle_type=vehicle_type,
                bbox=[x1, y1, x2, y2]
            )

            event = "detected"

            if track_id not in counted_tracks:
                counted_tracks[track_id] = {
                    "vehicle_type": vehicle_type,
                    "first_seen_frame": frame_number,
                    "first_seen_time": round(timestamp_seconds, 2),
                    "last_seen_frame": frame_number,
                    "last_seen_time": round(timestamp_seconds, 2)
                }
                event = "first_detected"
            else:
                counted_tracks[track_id]["last_seen_frame"] = frame_number
                counted_tracks[track_id]["last_seen_time"] = round(timestamp_seconds, 2)

                # If first counted as bus and later corrected as train,
                # update the final track label to train.
                if counted_tracks[track_id]["vehicle_type"] == "bus" and vehicle_type == "train":
                    counted_tracks[track_id]["vehicle_type"] = "train"

            report_rows.append([
                track_id,
                vehicle_type,
                frame_number,
                round(timestamp_seconds, 2),
                round(confidence, 4),
                round(x1, 2),
                round(y1, 2),
                round(x2, 2),
                round(y2, 2),
                event
            ])

            draw_box(
                frame=frame,
                bbox=[x1, y1, x2, y2],
                track_id=track_id,
                vehicle_type=vehicle_type,
                confidence=confidence
            )

            draw_trail(
                frame=frame,
                bbox=[x1, y1, x2, y2],
                track_id=track_id
            )

        vehicle_type_breakdown = {}

        for _, data in counted_tracks.items():
            vehicle_type = data["vehicle_type"]
            vehicle_type_breakdown[vehicle_type] = vehicle_type_breakdown.get(vehicle_type, 0) + 1

        progress_percent = (frame_number / total_frames) * 100 if total_frames > 0 else 0

        draw_summary_overlay(
            frame,
            total_count=len(counted_tracks),
            vehicle_type_breakdown=vehicle_type_breakdown,
            progress_percent=progress_percent
        )

        writer.write(frame)

    progress_bar.close()

    cap.release()
    writer.release()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    processing_duration_seconds = time.time() - start_time

    save_detection_report(report_rows, output_csv_path)

    summary = build_summary(
        counted_tracks=counted_tracks,
        processing_duration_seconds=processing_duration_seconds,
        total_frames=total_frames,
        fps=fps
    )

    summary["input_video_path"] = input_video_path
    summary["output_video_path"] = output_video_path
    summary["csv_report_path"] = output_csv_path
    summary["summary_json_path"] = output_summary_path
    summary["model"] = MODEL_PATH
    summary["image_size"] = IMAGE_SIZE
    summary["confidence_threshold"] = CONF_THRESHOLD
    summary["iou_threshold"] = IOU_THRESHOLD
    summary["frame_skip"] = FRAME_SKIP
    summary["device"] = "cuda:0" if torch.cuda.is_available() else "cpu"
    summary["class_correction"] = {
        "enabled": ENABLE_BUS_TO_TRAIN_HEURISTIC,
        "rule": "Large elongated bus detections are relabeled as train for drone top-view footage."
    }

    save_json(summary, output_summary_path)

    print("=" * 80)
    print("Processing complete.")
    print("Device used:", summary["device"])
    print("Total unique vehicles:", summary["total_vehicle_count"])
    print("Breakdown:", summary["vehicle_type_breakdown"])
    print("Processing duration seconds:", summary["processing_duration_seconds"])
    print("Output video:", output_video_path)
    print("CSV report:", output_csv_path)
    print("Summary JSON:", output_summary_path)
    print("=" * 80)

    return summary

Writing /content/smart_drone_colab/video_processor.py


# run_colab.py

In [13]:
%%writefile /content/smart_drone_colab/run_colab.py
import os

from config import VIDEO_1, VIDEO_2, OUTPUT_DIR
from file_utils import get_video_name_without_ext, ensure_dir
from video_processor import process_video


def run(video_path):
    video_name = get_video_name_without_ext(video_path)

    job_output_dir = os.path.join(OUTPUT_DIR, video_name)
    ensure_dir(job_output_dir)

    output_video_path = os.path.join(job_output_dir, f"{video_name}_processed.mp4")
    output_csv_path = os.path.join(job_output_dir, f"{video_name}_report.csv")
    output_summary_path = os.path.join(job_output_dir, f"{video_name}_summary.json")

    summary = process_video(
        input_video_path=video_path,
        output_video_path=output_video_path,
        output_csv_path=output_csv_path,
        output_summary_path=output_summary_path
    )

    return summary


if __name__ == "__main__":
    # Change VIDEO_1 to VIDEO_2 if needed
    summary = run(VIDEO_1)
    print(summary)

Writing /content/smart_drone_colab/run_colab.py


In [14]:
#text File Exist !!

import os

dataset_dir = "/content/drive/MyDrive/DATASET--ANTS"

print(os.listdir(dataset_dir))

video_1 = "/content/drive/MyDrive/DATASET--ANTS/Road Traffic - Dataset 01.mp4"
video_2 = "/content/drive/MyDrive/DATASET--ANTS/Road Traffic - Dataset 02.mp4"

print("Video 1 exists:", os.path.exists(video_1))
print("Video 2 exists:", os.path.exists(video_2))

['Road Traffic - Dataset 01.mp4', 'Road Traffic - Dataset 02.mp4', 'outputs']
Video 1 exists: True
Video 2 exists: True


# Run Processing on Dataset

In [15]:
#Dataset -01
import sys
sys.path.append("/content/smart_drone_colab")

# Remove previously loaded modules from sys.modules to force a fresh import
if 'config' in sys.modules:
    del sys.modules['config']
if 'file_utils' in sys.modules:
    del sys.modules['file_utils']
if 'report_generator' in sys.modules:
    del sys.modules['report_generator']
if 'visualizer' in sys.modules:
    del sys.modules['visualizer']
if 'video_processor' in sys.modules:
    del sys.modules['video_processor']
if 'run_colab' in sys.modules:
    del sys.modules['run_colab']

from run_colab import run
import config
from config import VIDEO_1

# Fix: Override the incorrect model path with the actual file found in /content/
config.MODEL_PATH = "/content/yolo26n.pt"

summary = run(VIDEO_1)

summary

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Loading model: /content/yolo26m.pt
Torch CUDA available: True
GPU name: Tesla T4
Current CUDA device: 0
Using YOLO device: cuda:0
Model loaded.
After model load | GPU memory | allocated: 84.16 MB | reserved: 92.00 MB
Video information:
FPS: 29.97002997002997
Width: 1280
Height: 720
Total frames: 305
Frame skip: 1
Image size: 640
Confidence threshold: 0.25
IOU threshold: 0.45
Starting video processing...


  0%|          | 0/305 [00:00<?, ?it/s]

requirements: Ultralytics requirement ['lap>=0.5.12'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 2 packages in 255ms
Prepared 1 package in 48ms
Installed 1 package in 5ms
 + lap==0.5.13

requirements: AutoUpdate success ✅ 0.7s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect



 99%|█████████▉| 302/305 [00:13<00:00, 22.59it/s]


Processing complete.
Device used: cuda:0
Total unique vehicles: 62
Breakdown: {'car': 57, 'truck': 2, 'train': 2, 'bus': 1}
Processing duration seconds: 13.38
Output video: /content/drive/MyDrive/DATASET--ANTS/outputs/Road_Traffic___Dataset_01/Road_Traffic___Dataset_01_processed.mp4
CSV report: /content/drive/MyDrive/DATASET--ANTS/outputs/Road_Traffic___Dataset_01/Road_Traffic___Dataset_01_report.csv
Summary JSON: /content/drive/MyDrive/DATASET--ANTS/outputs/Road_Traffic___Dataset_01/Road_Traffic___Dataset_01_summary.json


{'total_vehicle_count': 62,
 'vehicle_type_breakdown': {'car': 57, 'truck': 2, 'train': 2, 'bus': 1},
 'processing_duration_seconds': 13.38,
 'total_frames': 305,
 'fps': 29.97002997002997,
 'counting_method': 'Each unique ByteTrack tracking ID is counted only once.',
 'input_video_path': '/content/drive/MyDrive/DATASET--ANTS/Road Traffic - Dataset 01.mp4',
 'output_video_path': '/content/drive/MyDrive/DATASET--ANTS/outputs/Road_Traffic___Dataset_01/Road_Traffic___Dataset_01_processed.mp4',
 'csv_report_path': '/content/drive/MyDrive/DATASET--ANTS/outputs/Road_Traffic___Dataset_01/Road_Traffic___Dataset_01_report.csv',
 'summary_json_path': '/content/drive/MyDrive/DATASET--ANTS/outputs/Road_Traffic___Dataset_01/Road_Traffic___Dataset_01_summary.json',
 'model': '/content/yolo26m.pt',
 'image_size': 640,
 'confidence_threshold': 0.25,
 'iou_threshold': 0.45,
 'frame_skip': 1,
 'device': 'cuda:0',
 'class_correction': {'enabled': True,
  'rule': 'Large elongated bus detections are relabe

In [ ]:
import sys
sys.path.append("/content/smart_drone_colab")

if 'config' in sys.modules:
    del sys.modules['config']
if 'file_utils' in sys.modules:
    del sys.modules['file_utils']
if 'report_generator' in sys.modules:
    del sys.modules['report_generator']
if 'visualizer' in sys.modules:
    del sys.modules['visualizer']
if 'video_processor' in sys.modules:
    del sys.modules['video_processor']
if 'run_colab' in sys.modules:
    del sys.modules['run_colab']

from run_colab import run
import config
from config import VIDEO_2

summary = run(VIDEO_2)

summary

Loading model: /content/yolo26m.pt
Torch CUDA available: True
GPU name: Tesla T4
Current CUDA device: 0
Using YOLO device: cuda:0
Model loaded.
After model load | GPU memory | allocated: 205.42 MB | reserved: 220.00 MB
Video information:
FPS: 25.0
Width: 1280
Height: 720
Total frames: 51201
Frame skip: 1
Image size: 640
Confidence threshold: 0.25
IOU threshold: 0.45
Starting video processing...


  1%|          | 460/51201 [00:18<49:47, 16.98it/s]

KeyboardInterrupt: 

# View Generated Output Files

In [16]:
import os

output_dir = "/content/drive/MyDrive/DATASET--ANTS/outputs"

for root, dirs, files in os.walk(output_dir):
    level = root.replace(output_dir, "").count(os.sep)
    indent = " " * 4 * level
    print(f"{indent}{os.path.basename(root)}/")

    sub_indent = " " * 4 * (level + 1)
    for file in files:
        print(f"{sub_indent}{file}")

outputs/
    Road_Traffic___Dataset_01/
        Road_Traffic___Dataset_01_processed.mp4
        Road_Traffic___Dataset_01_summary.json
        Road_Traffic___Dataset_01_report.csv
        Road_Traffic___Dataset_01_processed_h264.mp4
    Road_Traffic___Dataset_02/
        Road_Traffic___Dataset_02_summary.json
        Road_Traffic___Dataset_02_processed.mp4


# CSV Report

In [17]:
import pandas as pd
import glob

csv_files = glob.glob("/content/drive/MyDrive/DATASET--ANTS/outputs/**/*.csv", recursive=True)

print(csv_files)

df = pd.read_csv(csv_files[0])
df.head(20)

['/content/drive/MyDrive/DATASET--ANTS/outputs/Road_Traffic___Dataset_01/Road_Traffic___Dataset_01_report.csv']


,track_id,vehicle_type,frame_number,timestamp_seconds,confidence,x1,y1,x2,y2,event
0,1,car,1,0.03,0.8719,679.32,0.00,714.50,20.83,first_detected
1,2,car,1,0.03,0.8435,154.42,0.00,227.58,26.40,first_detected
2,4,car,1,0.03,0.8298,620.04,0.21,654.25,22.54,first_detected
3,5,car,1,0.03,0.8058,46.50,0.00,125.43,29.94,first_detected
4,1,car,2,0.07,0.8719,679.32,0.00,714.50,20.83,detected
5,2,car,2,0.07,0.8436,154.42,0.00,227.58,26.40,detected
6,4,car,2,0.07,0.8298,620.04,0.21,654.25,22.54,detected
7,5,car,2,0.07,0.8059,46.51,0.00,125.42,29.94,detected
8,1,car,3,0.10,0.8849,678.94,0.11,715.83,21.99,detected
9,2,car,3,0.10,0.8367,157.04,0.00,229.85,26.30,detected


# Summary JSON

In [18]:
import json
import glob

summary_files = glob.glob("/content/drive/MyDrive/DATASET--ANTS/outputs/**/*.json", recursive=True)

print(summary_files)

with open(summary_files[0], "r") as f:
    summary = json.load(f)

summary

['/content/drive/MyDrive/DATASET--ANTS/outputs/Road_Traffic___Dataset_01/Road_Traffic___Dataset_01_summary.json', '/content/drive/MyDrive/DATASET--ANTS/outputs/Road_Traffic___Dataset_02/Road_Traffic___Dataset_02_summary.json']


{'total_vehicle_count': 62,
 'vehicle_type_breakdown': {'car': 57, 'truck': 2, 'train': 2, 'bus': 1},
 'processing_duration_seconds': 13.38,
 'total_frames': 305,
 'fps': 29.97002997002997,
 'counting_method': 'Each unique ByteTrack tracking ID is counted only once.',
 'input_video_path': '/content/drive/MyDrive/DATASET--ANTS/Road Traffic - Dataset 01.mp4',
 'output_video_path': '/content/drive/MyDrive/DATASET--ANTS/outputs/Road_Traffic___Dataset_01/Road_Traffic___Dataset_01_processed.mp4',
 'csv_report_path': '/content/drive/MyDrive/DATASET--ANTS/outputs/Road_Traffic___Dataset_01/Road_Traffic___Dataset_01_report.csv',
 'summary_json_path': '/content/drive/MyDrive/DATASET--ANTS/outputs/Road_Traffic___Dataset_01/Road_Traffic___Dataset_01_summary.json',
 'model': '/content/yolo26m.pt',
 'image_size': 640,
 'confidence_threshold': 0.25,
 'iou_threshold': 0.45,
 'frame_skip': 1,
 'device': 'cuda:0',
 'class_correction': {'enabled': True,
  'rule': 'Large elongated bus detections are relabe

# Convert Output Video for Browser Preview

In [19]:
import glob
import os

processed_videos = glob.glob("/content/drive/MyDrive/DATASET--ANTS/outputs/**/*_processed.mp4", recursive=True)

input_video = processed_videos[0]
converted_video = input_video.replace("_processed.mp4", "_processed_h264.mp4")

print("Input:", input_video)
print("Converted:", converted_video)

!ffmpeg -y -i "$input_video" -vcodec libx264 -acodec aac "$converted_video"

Input: /content/drive/MyDrive/DATASET--ANTS/outputs/Road_Traffic___Dataset_01/Road_Traffic___Dataset_01_processed.mp4
Converted: /content/drive/MyDrive/DATASET--ANTS/outputs/Road_Traffic___Dataset_01/Road_Traffic___Dataset_01_processed_h264.mp4
ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --e

# Display Processed Video in Colab

In [20]:
from IPython.display import HTML
from base64 import b64encode

video_path = converted_video

mp4 = open(video_path, "rb").read()
data_url = "data:video/mp4;base64," + b64encode(mp4).decode()

HTML(f"""
<video width=800 controls>
    <source src="{data_url}" type="video/mp4">
</video>
""")